# 05 - FastSurfer baseline, volume agreement and boundary metrics

No training is done here. The notebook:

1. re-runs inference with the checkpoints saved by notebook 01 (random init. and
   SSL pretrained, 25 and 104 labeled scans), saves the predictions and measures
   structure volumes and boundary distances;
2. runs FastSurfer (segmentation module only) on all scans;
3. scores FastSurfer and SynthSeg (outputs of notebook 04) with the same metrics;
4. writes summary tables: Dice, volume agreement with FreeSurfer `aseg`, and
   boundary distances (HD95, ASSD) after keeping the largest connected component.

Long steps skip finished scans, so they can be interrupted and re-run.
A T4 GPU runtime is enough.

## 1. Settings, file lists and folds

In [ ]:
import os, glob, json, time, shutil, subprocess, sys
import numpy as np
import pandas as pd

try:
    import monai
except ImportError:                                             # Colab does not ship MONAI
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'monai'], check=True)
import nibabel as nib

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Drive not mounted here:', e)

# same folders as notebook 01
IMAGE_DIR = '/content/drive/MyDrive/OASIS3/images'
LABEL_DIR = '/content/drive/MyDrive/OASIS3/labels'
ROOT_DIR  = '/content/drive/MyDrive/brain_seg_supervised'       # outputs of notebooks 01 and 04
META_CSV  = '/content/drive/MyDrive/OASIS3/metadata.csv'         # optional, same file as notebook 02
TMP_DIR   = '/content/tmp05'
CV_DIR  = os.path.join(ROOT_DIR, 'cv_results')
SSL_DIR = os.path.join(ROOT_DIR, 'ssl_results')
SS_SEG_DIR = os.path.join(ROOT_DIR, 'synthseg', 'seg')          # SynthSeg outputs from notebook 04
OUT_DIR  = os.path.join(ROOT_DIR, 'extra_analysis')             # everything from this notebook
PRED_DIR = os.path.join(OUT_DIR, 'pred')                        # saved predictions of our models
FS_SEG_DIR = os.path.join(OUT_DIR, 'fastsurfer_seg')            # FastSurfer outputs
for d in (OUT_DIR, PRED_DIR, FS_SEG_DIR, TMP_DIR):
    os.makedirs(d, exist_ok=True)

SEED, FOLDS = 42, 5
STD_LABEL_VALUES = np.array([0,2,3,4,7,8,10,11,12,13,14,15,16,17,28,31,
                             41,42,43,46,47,49,50,51,52,53,60,63])
KEEP = [int(v) for v in STD_LABEL_VALUES[1:]]                   # the 27 structures (FreeSurfer values)
NAMES = {2:'Left cerebral white matter',3:'Left cerebral cortex',4:'Left lateral ventricle',
         7:'Left cerebellum white matter',8:'Left cerebellum cortex',10:'Left thalamus',11:'Left caudate',
         12:'Left putamen',13:'Left pallidum',14:'Third ventricle',15:'Fourth ventricle',16:'Brainstem',
         17:'Left hippocampus',28:'Left ventral DC',31:'Left choroid plexus',
         41:'Right cerebral white matter',42:'Right cerebral cortex',43:'Right lateral ventricle',
         46:'Right cerebellum white matter',47:'Right cerebellum cortex',49:'Right thalamus',50:'Right caudate',
         51:'Right putamen',52:'Right pallidum',53:'Right hippocampus',60:'Right ventral DC',63:'Right choroid plexus'}

image_files = sorted(glob.glob(f'{IMAGE_DIR}/**/*.nii.gz*', recursive=True))
label_files = sorted(glob.glob(f'{LABEL_DIR}/**/*.nii.gz*', recursive=True))
assert len(image_files) == len(label_files) > 0, 'images and labels not found - check IMAGE_DIR / LABEL_DIR'
n = len(image_files)
perm = np.random.RandomState(SEED).permutation(n)               # same test folds as notebook 01
fold_id = np.zeros(n, dtype=int)
for k in range(FOLDS):
    fold_id[perm[k::FOLDS]] = k
name_of = lambda p: os.path.basename(p).split('.nii')[0]
print(f'{n} subjects | test-fold sizes: {np.bincount(fold_id).tolist()} | outputs -> {OUT_DIR}')

## 2. Metric functions

For every structure we store: Dice, volume (ml) of the prediction and of the
FreeSurfer reference, and two boundary distances computed after keeping only the
largest connected component of the predicted structure (this removes stray voxels,
which made the raw Hausdorff distance unusable):
the 95th-percentile Hausdorff distance (HD95) and the average symmetric surface
distance (ASSD), both in mm (the images are 1 mm isotropic).

In [ ]:
from scipy import ndimage as ndi

_LUT = np.zeros(256, dtype=np.int64)                            # FreeSurfer value -> class index 0..27
for _i, _v in enumerate(STD_LABEL_VALUES):
    _LUT[_v] = _i
_S26 = np.ones((3, 3, 3), dtype=bool)


def fs_to_index(a):
    """FreeSurfer label values -> class indices 0..27 (anything else -> 0).
    Cortical parcels (1000-2999, as written by FastSurfer) are merged into cortex."""
    a = np.asarray(a).astype(np.int64).copy()
    a[(a >= 1000) & (a < 2000)] = 3
    a[(a >= 2000) & (a < 3000)] = 42
    a[(a < 0) | (a > 255)] = 0
    return _LUT[a].astype(np.uint8)


def largest_component(m):
    lab, k = ndi.label(m, structure=_S26)
    if k <= 1:
        return m
    sizes = np.bincount(lab.ravel()); sizes[0] = 0
    return lab == sizes.argmax()


def _surface(m):
    return m & ~ndi.binary_erosion(m)


def boundary_distances(p, g):
    """HD95 and ASSD (in voxels = mm) between two binary masks."""
    if not p.any() or not g.any():
        return np.nan, np.nan
    sp, sg = _surface(p), _surface(g)
    d_pg = ndi.distance_transform_edt(~sg)[sp]                  # predicted surface -> reference surface
    d_gp = ndi.distance_transform_edt(~sp)[sg]                  # reference surface -> predicted surface
    hd95 = max(np.percentile(d_pg, 95), np.percentile(d_gp, 95))
    assd = (d_pg.sum() + d_gp.sum()) / (len(d_pg) + len(d_gp))
    return float(hd95), float(assd)


def subject_metrics(pred, ref, voxel_ml=0.001):
    """pred, ref: class-index volumes (0..27) on the same grid. One row per structure."""
    op, og = ndi.find_objects(pred, max_label=27), ndi.find_objects(ref, max_label=27)
    rows = []
    for c in range(1, 28):
        sp, sg = op[c - 1], og[c - 1]
        row = dict(label=KEEP[c - 1], dice=np.nan, dice_lcc=np.nan, vol_pred_ml=0.0, vol_lcc_ml=0.0,
                   vol_ref_ml=0.0, hd95_lcc=np.nan, assd_lcc=np.nan)
        if sg is None:                                          # structure absent in the reference
            row['vol_pred_ml'] = float((pred == c).sum()) * voxel_ml
            rows.append(row); continue
        if sp is None:
            sl = sg
        else:
            sl = tuple(slice(max(min(a.start, b.start) - 1, 0), max(a.stop, b.stop) + 1) for a, b in zip(sp, sg))
        p, g = pred[sl] == c, ref[sl] == c
        vp, vg = int(p.sum()), int(g.sum())
        row['vol_pred_ml'], row['vol_ref_ml'] = vp * voxel_ml, vg * voxel_ml
        row['dice'] = 2.0 * int((p & g).sum()) / (vp + vg)
        if vp > 0:
            pl = largest_component(p)
            vl = int(pl.sum())
            row['vol_lcc_ml'] = vl * voxel_ml
            row['dice_lcc'] = 2.0 * int((pl & g).sum()) / (vl + vg)
            row['hd95_lcc'], row['assd_lcc'] = boundary_distances(pl, g)
        else:
            row['dice_lcc'] = 0.0
        rows.append(row)
    return rows


def metrics_path(method):
    return os.path.join(OUT_DIR, f'metrics_{method}.csv')


def done_subjects(method):
    p = metrics_path(method)
    return set(pd.read_csv(p).subject.unique()) if os.path.exists(p) else set()


def append_rows(method, subject, fold, rows):
    df = pd.DataFrame(rows); df.insert(0, 'fold', fold); df.insert(0, 'subject', subject)
    p = metrics_path(method)
    df.to_csv(p, mode='a', header=not os.path.exists(p), index=False)

print('metric functions ready')

## 3. Our trained models: find the checkpoints

The models compared here are the randomly initialized and the pretrained model at
25 and at 104 labeled subjects. This cell only looks for their checkpoint files.
If any are missing, the `.pth` files that do exist are listed.

In [ ]:
import torch
from monai.networks.nets import UNet
from monai.networks.layers import Norm
from monai.inferers import sliding_window_inference
from monai.transforms import (Compose, LoadImaged, EnsureTyped, Lambdad, EnsureChannelFirstd,
                              Orientationd, Spacingd, NormalizeIntensityd)

MODELS = [('scratch', 25), ('ssl', 25), ('scratch', 104), ('ssl', 104)]
CKPT_NAME = 'best_fold{k}_{arm}_n{n}.pth'
CKPT_DIRS = [SSL_DIR, CV_DIR]
NUM_CLASSES, PATCH_SIZE = 28, (128, 128, 128)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def find_ckpt(k, arm, n_lab):
    for d in CKPT_DIRS:
        p = os.path.join(d, CKPT_NAME.format(k=k, arm=arm, n=n_lab))
        if os.path.exists(p):
            return p
    return None


CKPT = {(arm, n_lab, k): find_ckpt(k, arm, n_lab) for arm, n_lab in MODELS for k in range(FOLDS)}
missing = [key for key, p in CKPT.items() if p is None]
for (arm, n_lab) in MODELS:
    found = [os.path.basename(CKPT[(arm, n_lab, k)]) if CKPT[(arm, n_lab, k)] else 'MISSING' for k in range(FOLDS)]
    print(f'{arm:8s} n={n_lab:<4d}', found)
print(f'\ndevice: {device} | checkpoints found: {len(CKPT) - len(missing)} of {len(CKPT)}')
if missing:
    print('\nSome checkpoints were not found. Files ending in .pth that exist:')
    for d in CKPT_DIRS:
        fs = sorted(glob.glob(os.path.join(d, '*.pth')))
        print(f'  {d}: {len(fs)} files')
        for f in fs[:60]:
            print('     ', os.path.basename(f))

## 4. Our trained models: inference, saved predictions and metrics

Same preprocessing and sliding-window settings as the evaluation in notebook 01.
About 15-30 s per scan and model on a T4 GPU. The last part checks the recomputed
Dice against the values stored by notebook 01.

In [ ]:
def remap_labels(x):
    if not isinstance(x, torch.Tensor):
        x = torch.as_tensor(np.asarray(x))
    idx = np.clip(x.detach().cpu().numpy().astype(np.int64), 0, 255)
    out = x.clone().float()
    out.copy_(torch.from_numpy(_LUT[idx].astype(np.float32)))
    return out


eval_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureTyped(keys=["image", "label"], data_type="tensor", track_meta=True),
    Lambdad(keys="label", func=remap_labels),
    EnsureChannelFirstd(keys=["image", "label"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    Spacingd(keys=["image", "label"], pixdim=(1.0, 1.0, 1.0), mode=("bilinear", "nearest")),
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
    EnsureTyped(keys=["image", "label"]),
])


def build_model():
    return UNet(spatial_dims=3, in_channels=1, out_channels=NUM_CLASSES,
                channels=(32, 64, 128, 256, 320), strides=(2, 2, 2, 2),
                num_res_units=2, norm=Norm.INSTANCE, dropout=0.1).to(device)


def load_model(path):
    m = build_model()
    sd = torch.load(path, map_location=device, weights_only=False)
    m.load_state_dict(sd['model_state_dict'] if isinstance(sd, dict) and 'model_state_dict' in sd else sd)
    return m.eval()


method_of = lambda arm, n_lab: f'{arm}_n{n_lab}'
t_start, n_done = time.time(), 0
for k in range(FOLDS):
    avail = [(arm, n_lab) for arm, n_lab in MODELS if CKPT[(arm, n_lab, k)]]
    todo = {m: [i for i in np.where(fold_id == k)[0] if name_of(image_files[i]) not in done_subjects(method_of(*m))]
            for m in avail}
    subj = sorted(set(i for v in todo.values() for i in v))
    if not subj:
        print(f'fold {k}: nothing to do'); continue
    nets = {m: load_model(CKPT[(m[0], m[1], k)]) for m in avail if todo[m]}
    print(f'fold {k}: {len(subj)} scans x {len(nets)} models')
    for i in subj:
        name = name_of(image_files[i])
        data = eval_transforms({'image': image_files[i], 'label': label_files[i]})
        img = data['image'][None].to(device)
        ref = np.asarray(data['label'][0]).astype(np.uint8)
        for m, net in nets.items():
            if i not in todo[m]:
                continue
            with torch.no_grad(), torch.autocast(device_type=device.type, enabled=device.type == 'cuda'):
                logits = sliding_window_inference(img, PATCH_SIZE, 2, net, overlap=0.5)
            pred = logits.argmax(1)[0].cpu().numpy().astype(np.uint8)
            del logits
            os.makedirs(os.path.join(PRED_DIR, method_of(*m)), exist_ok=True)
            np.savez_compressed(os.path.join(PRED_DIR, method_of(*m), name + '.npz'), pred=pred)
            append_rows(method_of(*m), name, k, subject_metrics(pred, ref))
            n_done += 1
        if n_done and n_done % 20 < len(nets):
            print(f'   {n_done} done, {(time.time() - t_start) / max(n_done, 1):.0f} s per scan and model')
    del nets
    if device.type == 'cuda':
        torch.cuda.empty_cache()

# recomputed Dice against the values stored by notebook 01
for arm, n_lab in MODELS:
    p = metrics_path(method_of(arm, n_lab))
    if not os.path.exists(p):
        continue
    df = pd.read_csv(p)
    for k in range(FOLDS):
        jp = os.path.join(SSL_DIR, f'results_fold{k}_{arm}_n{n_lab}.json')
        d = df[df.fold == k]
        if len(d) and os.path.exists(jp):
            mine = d.groupby('label').dice.mean().mean()
            stored = float(np.nanmean(json.load(open(jp))['per_class_dice']))
            flag = '' if abs(mine - stored) < 0.003 else '   <-- differs'
            print(f'{arm:8s} n={n_lab:<4d} fold {k}: recomputed {mine:.4f} | stored {stored:.4f}{flag}')

## 5. FastSurfer: install

FastSurfer (Henschel et al., 2020) is used as provided, with no training on this cohort.
Only the segmentation module is run (`--seg_only`); the corpus callosum step is
switched off (`--no_cc`) because it is not needed for the 27 structures.

In [ ]:
FASTSURFER_HOME = '/content/FastSurfer'
if not os.path.exists(os.path.join(FASTSURFER_HOME, 'run_fastsurfer.sh')):
    r = subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', 'stable',
                        'https://github.com/Deep-MI/FastSurfer.git', FASTSURFER_HOME])
    if r.returncode != 0:
        subprocess.run(['git', 'clone', '-q', '--depth', '1',
                        'https://github.com/Deep-MI/FastSurfer.git', FASTSURFER_HOME], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'yacs', 'h5py', 'nibabel', 'torchio',
                'lapy', 'simpleitk', 'scikit-image'])
_script = open(os.path.join(FASTSURFER_HOME, 'run_fastsurfer.sh')).read()
FS_OPTS = [f for f in ['--no_cereb', '--no_hypothal', '--no_biasfield', '--no_cc', '--allow_root'] if f in _script]
r = subprocess.run(['git', '-C', FASTSURFER_HOME, 'log', '-1', '--format=%h %cd', '--date=short'],
                   capture_output=True, text=True)
print('FastSurfer commit:', r.stdout.strip(), '| options used:', FS_OPTS, '| GPU:', torch.cuda.is_available())


def fs_seg_of(p):
    return os.path.join(FS_SEG_DIR, name_of(p) + '_fastsurfer.mgz')


def run_fastsurfer(img_path):
    """Segment one scan. Returns (ok, log)."""
    sid = name_of(img_path)
    local = os.path.join(TMP_DIR, sid + '.nii.gz')
    shutil.copy(img_path, local)
    sd = os.path.join(TMP_DIR, 'fs_out')
    cmd = ['bash', os.path.join(FASTSURFER_HOME, 'run_fastsurfer.sh'), '--t1', local, '--sid', sid, '--sd', sd,
           '--seg_only', '--py', sys.executable, '--threads', str(os.cpu_count())] + FS_OPTS
    if not torch.cuda.is_available():
        cmd += ['--device', 'cpu']
    r = subprocess.run(cmd, capture_output=True, text=True, env=dict(os.environ, FASTSURFER_HOME=FASTSURFER_HOME))
    out = os.path.join(sd, sid, 'mri', 'aparc.DKTatlas+aseg.deep.mgz')
    # aparc.DKTatlas+aseg.deep.mgz is written before FastSurfer's optional later steps,
    # so it is kept even if one of those steps fails.
    ok = False
    if os.path.exists(out):
        try:
            ok = len(nib.load(out).shape) == 3
        except Exception:
            ok = False
    if ok:
        shutil.copy(out, fs_seg_of(img_path))
    shutil.rmtree(os.path.join(sd, sid), ignore_errors=True)
    if os.path.exists(local):
        os.remove(local)
    return ok, (r.stdout[-2500:] + '\n' + r.stderr[-2500:])

## 6. FastSurfer: test on one scan

Checks the installation before the full run. The log is printed if it fails.

In [ ]:
t0 = time.time()
test_img = image_files[0]
if os.path.exists(fs_seg_of(test_img)):
    print('already segmented:', fs_seg_of(test_img))
else:
    ok, log = run_fastsurfer(test_img)
    print(f'ok: {ok} | {(time.time() - t0) / 60:.1f} min')
    if not ok:
        print(log)
if os.path.exists(fs_seg_of(test_img)):
    vals = np.unique(np.asarray(nib.load(fs_seg_of(test_img)).dataobj))
    have = [v for v in KEEP if v in vals or (v == 3 and (vals >= 1000).any()) or (v == 42 and (vals >= 2000).any())]
    print(f'{len(vals)} labels in the output | {len(have)} of our 27 structures present | '
          f'missing: {[NAMES[v] for v in KEEP if v not in have]}')

## 7. FastSurfer: all scans

About 1-2 min per scan on a T4 GPU.

In [ ]:
todo = [p for p in image_files if not os.path.exists(fs_seg_of(p))]
print(f'{len(todo)} of {n} scans still to segment')
t0 = time.time()
for j, p in enumerate(todo):
    ok, log = run_fastsurfer(p)
    if not ok:
        print('FAILED on', name_of(p)); print(log); break
    if (j + 1) % 10 == 0 or j + 1 == len(todo):
        print(f'{j + 1}/{len(todo)} | {(time.time() - t0) / (j + 1) / 60:.1f} min per scan')

## 8. Metrics for FastSurfer and SynthSeg

Same metrics as for our models, against the same FreeSurfer reference.
SynthSeg outputs come from notebook 04 (skipped if not found).
Cortical parcels in the FastSurfer output are merged into left and right cortex.

In [ ]:
from nibabel.processing import resample_from_to


def seg_on_label_grid(seg_path, ref_img):
    s = nib.load(seg_path)
    if s.shape != ref_img.shape or not np.allclose(s.affine, ref_img.affine, atol=1e-3):
        s = resample_from_to(s, ref_img, order=0)               # nearest neighbour onto the label grid
    return np.rint(np.asarray(s.dataobj)).astype(np.int64)


BASELINES = {'fastsurfer': lambda p: os.path.join(FS_SEG_DIR, name_of(p) + '_fastsurfer.mgz'),
             'synthseg': lambda p: os.path.join(SS_SEG_DIR, name_of(p) + '_synthseg.nii.gz')}
for method, seg_of in BASELINES.items():
    done = done_subjects(method)
    todo = [i for i in range(n) if name_of(image_files[i]) not in done and os.path.exists(seg_of(image_files[i]))]
    print(f'{method}: {len(done)} scored, {len(todo)} to score, '
          f'{n - len(done) - len(todo)} without a segmentation yet')
    t0 = time.time()
    for j, i in enumerate(todo):
        ref_img = nib.load(label_files[i])
        ref = fs_to_index(np.asarray(ref_img.dataobj))
        pred = fs_to_index(seg_on_label_grid(seg_of(image_files[i]), ref_img))
        append_rows(method, name_of(image_files[i]), int(fold_id[i]), subject_metrics(pred, ref))
        if (j + 1) % 25 == 0:
            print(f'   {j + 1}/{len(todo)} | {(time.time() - t0) / (j + 1):.0f} s per scan')

## 9. Summary tables

Prints the results and saves them as `table_*.csv` in `extra_analysis`.
The `metrics_*.csv` files in the same folder are per subject and stay private
(OASIS data use agreement); the `table_*.csv` files are summaries only.

In [ ]:
from scipy import stats

RHO = 1.0 / FOLDS + 30.0 / 104.0                                # variance correction for dependent folds


def corrected_p(d):
    d = np.asarray(d, dtype=float)
    if len(d) != FOLDS or np.isnan(d).any() or d.var(ddof=1) == 0:
        return np.nan
    return float(2 * stats.t.sf(abs(d.mean() / np.sqrt(RHO * d.var(ddof=1))), FOLDS - 1))


def fold_values(df, col, labels):
    """Per fold: mean over structures of the mean over subjects."""
    d = df[df.label.isin(labels)]
    return d.groupby(['fold', 'label'])[col].mean().groupby('fold').mean().reindex(range(FOLDS)).values


LABELS_OF = {'scratch_n25': 'Random init., n=25', 'ssl_n25': 'SSL pretrained, n=25',
             'scratch_n104': 'Random init., n=104', 'ssl_n104': 'SSL pretrained, n=104',
             'synthseg': 'SynthSeg', 'fastsurfer': 'FastSurfer'}
M = {m: pd.read_csv(metrics_path(m)) for m in LABELS_OF if os.path.exists(metrics_path(m))}
print('methods with metrics:', {m: int(d.subject.nunique()) for m, d in M.items()}, '\n')
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)

# ---- A. Dice: each baseline against our models, on the structures the baseline segments
json_dice = {}
for n_lab in [10, 25, 50, 104]:
    for arm in ['scratch', 'ssl']:
        pc = []
        for k in range(FOLDS):
            jp = os.path.join(SSL_DIR, f'results_fold{k}_{arm}_n{n_lab}.json')
            if os.path.exists(jp):
                pc.append(np.asarray(json.load(open(jp))['per_class_dice'], dtype=float))
        if len(pc) == FOLDS:
            json_dice[(arm, n_lab)] = np.vstack(pc)             # folds x 27
rows = []
for b in ['fastsurfer', 'synthseg']:
    if b not in M or M[b].subject.nunique() < n:
        continue
    seen = M[b].groupby('label').vol_pred_ml.apply(lambda v: (v > 0).mean())
    labs = [v for v in KEEP if seen.get(v, 0) > 0.5]
    idx = [KEEP.index(v) for v in labs]
    bf = fold_values(M[b], 'dice', labs)
    rows.append(dict(baseline=LABELS_OF[b], n_structures=len(labs), method=LABELS_OF[b], n_labeled='-',
                     dice_mean=bf.mean(), dice_sd=bf.std(ddof=1), baseline_minus_model=np.nan, p_corr=np.nan))
    for (arm, n_lab), pc in json_dice.items():
        v = np.nanmean(pc[:, idx], axis=1)
        rows.append(dict(baseline=LABELS_OF[b], n_structures=len(labs),
                         method='Random init.' if arm == 'scratch' else 'SSL pretrained', n_labeled=n_lab,
                         dice_mean=v.mean(), dice_sd=v.std(ddof=1), baseline_minus_model=(bf - v).mean(),
                         p_corr=corrected_p(bf - v)))
if rows:
    A = pd.DataFrame(rows); A.to_csv(os.path.join(OUT_DIR, 'table_dice_vs_baselines.csv'), index=False)
    print('A. Dice of each baseline against our models (mean +/- SD over 5 test folds)\n')
    print(A.round(4).to_string(index=False), '\n')

# ---- B. Volume agreement with FreeSurfer
rows = []
for m, df in M.items():
    for v in KEEP:
        d = df[(df.label == v) & (df.vol_ref_ml > 0)]
        if len(d) < 10 or d.vol_pred_ml.std() == 0:
            continue
        pct = 100.0 * (d.vol_pred_ml - d.vol_ref_ml) / d.vol_ref_ml
        pcl = 100.0 * (d.vol_lcc_ml - d.vol_ref_ml) / d.vol_ref_ml
        rows.append(dict(method=LABELS_OF[m], structure=NAMES[v], n=len(d), ref_ml=d.vol_ref_ml.mean(),
                         pred_ml=d.vol_pred_ml.mean(), pearson_r=stats.pearsonr(d.vol_pred_ml, d.vol_ref_ml)[0],
                         mean_pct_diff=pct.mean(), mean_abs_pct_diff=pct.abs().mean(),
                         clean_ml=d.vol_lcc_ml.mean(),
                         clean_pearson_r=stats.pearsonr(d.vol_lcc_ml, d.vol_ref_ml)[0] if d.vol_lcc_ml.std() > 0 else np.nan,
                         clean_mean_pct_diff=pcl.mean(), clean_mean_abs_pct_diff=pcl.abs().mean()))
if rows:
    B = pd.DataFrame(rows); B.to_csv(os.path.join(OUT_DIR, 'table_volume_by_structure.csv'), index=False)
    Bs = B.groupby('method', sort=False).agg(structures=('structure', 'count'), median_r=('pearson_r', 'median'),
                                             min_r=('pearson_r', 'min'), mean_abs_pct_diff=('mean_abs_pct_diff', 'mean'),
                                             mean_pct_diff=('mean_pct_diff', 'mean'),
                                             clean_median_r=('clean_pearson_r', 'median'),
                                             clean_min_r=('clean_pearson_r', 'min'),
                                             clean_mean_abs_pct_diff=('clean_mean_abs_pct_diff', 'mean'),
                                             clean_mean_pct_diff=('clean_mean_pct_diff', 'mean')).reset_index()
    Bs.to_csv(os.path.join(OUT_DIR, 'table_volume_summary.csv'), index=False)
    print('B. Volume agreement with FreeSurfer (summary over structures; "clean" = largest component kept)\n')
    print(Bs.round(3).to_string(index=False), '\n')

# ---- C. Boundary distances after largest-component clean-up
rows, per_fold = [], {}
for m, df in M.items():
    labs = [v for v in KEEP if (df[df.label == v].vol_pred_ml > 0).mean() > 0.5]
    hd, asd = fold_values(df, 'hd95_lcc', labs), fold_values(df, 'assd_lcc', labs)
    dr, dl = fold_values(df, 'dice', labs), fold_values(df, 'dice_lcc', labs)
    per_fold[m] = dict(hd=hd, asd=asd)
    d = df[df.label.isin(labs) & (df.vol_ref_ml > 0)]
    rows.append(dict(method=LABELS_OF[m], n_structures=len(labs), hd95_mean=np.nanmean(hd), hd95_sd=np.nanstd(hd, ddof=1),
                     assd_mean=np.nanmean(asd), assd_sd=np.nanstd(asd, ddof=1), dice=np.nanmean(dr),
                     dice_after_cleanup=np.nanmean(dl), empty_predictions=int(d.hd95_lcc.isna().sum())))
if rows:
    C = pd.DataFrame(rows); C.to_csv(os.path.join(OUT_DIR, 'table_boundary_summary.csv'), index=False)
    print('C. Boundary distances in mm, largest component kept (mean +/- SD over 5 test folds)\n')
    print(C.round(4).to_string(index=False), '\n')
    Cs = pd.concat([df[df.vol_ref_ml > 0].groupby('label')[['hd95_lcc', 'assd_lcc', 'dice', 'dice_lcc']].mean()
                    .assign(method=LABELS_OF[m]).reset_index() for m, df in M.items()])
    Cs.insert(1, 'structure', Cs.label.map(NAMES))
    Cs.to_csv(os.path.join(OUT_DIR, 'table_boundary_by_structure.csv'), index=False)
    rows = []
    for n_lab in [25, 104]:
        a, b = f'scratch_n{n_lab}', f'ssl_n{n_lab}'
        if a in per_fold and b in per_fold:
            for key, nm in [('hd', 'HD95'), ('asd', 'ASSD')]:
                dlt = per_fold[a][key] - per_fold[b][key]
                rows.append(dict(n_labeled=n_lab, metric=nm, random_init=per_fold[a][key].mean(),
                                 ssl=per_fold[b][key].mean(), reduction_with_ssl=dlt.mean(), p_corr=corrected_p(dlt),
                                 folds_better=int((dlt > 0).sum())))
    if rows:
        C2 = pd.DataFrame(rows); C2.to_csv(os.path.join(OUT_DIR, 'table_boundary_ssl_vs_random.csv'), index=False)
        print('   Pretrained against randomly initialized (positive reduction = pretraining is better)\n')
        print(C2.round(4).to_string(index=False), '\n')

# ---- D. Scanner check (cognitively normal participants only)
if os.path.exists(META_CSV) and M:
    meta = pd.read_csv(META_CSV).dropna(subset=['image_name'])
    meta['subject'] = meta.image_name.str.replace(r'\.nii(\.gz)?$', '', regex=True)
    meta['CDR'] = pd.to_numeric(meta.CDR_global, errors='coerce')
    rows = []
    for m, df in M.items():
        labs = [v for v in KEEP if (df[df.label == v].vol_pred_ml > 0).mean() > 0.5]
        s = df[df.label.isin(labs)].groupby('subject').dice.mean().reset_index().merge(meta, on='subject')
        cn = s[(s.CDR == 0)]
        tt, bg = cn[cn.scanner_model == 'TrioTim'].dice, cn[cn.scanner_model == 'Biograph_mMR'].dice
        if len(tt) > 3 and len(bg) > 3:
            rows.append(dict(method=LABELS_OF[m], trio_n=len(tt), trio_dice=tt.mean(), biograph_n=len(bg),
                             biograph_dice=bg.mean(), difference=bg.mean() - tt.mean(),
                             mann_whitney_p=stats.mannwhitneyu(tt, bg)[1]))
    if rows:
        D = pd.DataFrame(rows); D.to_csv(os.path.join(OUT_DIR, 'table_scanner_check.csv'), index=False)
        print('D. Dice by scanner, cognitively normal participants only\n')
        print(D.round(4).to_string(index=False), '\n')
else:
    print('D. metadata CSV not found - scanner check skipped\n')

print('Saved in', OUT_DIR, ':')
for f in sorted(glob.glob(os.path.join(OUT_DIR, 'table_*.csv'))):
    print('  ', os.path.basename(f))